### Decision Trees - ID3

Given patient, age, blood pressure(high/low), cholesterol(high/normal), diagnosis(healthy,sick)

- calculate entropy for target variable
- calculate information gain for each feature
- using ID3, which feature is the root node? 
- build decision tree
- predict: 50 year old, low bp, normal cholesterol
- without scikit

In [ ]:
import pandas as pd
import numpy as np
import math

data = pd.DataFrame({
    'outlook': ['s','s','o','r','r','r','o','s','s','r','s','o','o','r'],
    'temp': [85,80,83,70,68,65,64,72,69,75,75,72,81,71],
    'humidity': [85,90,78,96,80,70,65,95,70,80,70,90,75,80],
    'wind': ['w','s','w','w','w','s','s','w','w','w','s','s','w','s'],
    'play': [0,0,1,1,1,0,1,0,1,1,1,1,1,0]
})

# 2. Convert numerical attributes into categories
data['temp'] = np.where(data['temp'] > 75, 'h', 'l')
data['humidity'] = np.where(data['humidity'] > 80, 'h', 'l')

features = ['outlook', 'temp', 'humidity', 'wind']
target = 'play'


def entropy(feature):
    c1 = len((feature == 0)) / len(feature)
    c2 = len((feature == 1)) / len(feature)

    if c1 > 0 and c2 > 0:
        r = c1 * math.log2(c1) + c2 * math.log2(c2)
    elif c1 > 0:
        r = c1 * math.log2(c1)
    else:
        r = c2 * math.log2(c2)

    return -r


def information_gain(data, feature):
    total_entropy = entropy(data["play"])
    weighted_entropy = 0
    total = len(data)

    for value in data[feature].unique():
        subset = data[data[feature] == value]
        p = len(subset) / total

        weighted_entropy += p * entropy(subset["play"])

    return total_entropy - weighted_entropy


def build_tree(df, features, target):

    # If all target values are identical
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no features remain, return majority class
    if len(features) == 0:
        return df[target].mode().iloc[0]

    gains = {}

    for feature in features:
        gains[feature] = information_gain(df, feature)

    best_feature = max(gains, key=gains.get)

    tree = {best_feature: {}}

    # Create branches for each feature value
    for value, subset in df.groupby(best_feature):
        remaining_features = [f for f in features if f != best_feature]
        subset = subset.drop(columns=[best_feature])

        tree[best_feature][value] = build_tree(subset, remaining_features, target)

    return tree


# 6. Classify a new sample
def classify(tree, sample, default_class):

    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample[feature]

    if value in tree[feature]:
        return classify(tree[feature][value], sample, default_class)

    return default_class


# 7. Build and print the decision tree
tree = build_tree(data, features, target)

print("Generated ID3 Decision Tree:")

from pprint import pprint
pprint(tree, sort_dicts=False)


# 8. Classify a new sample
new_sample = pd.Series({
    'outlook': 's',
    'temp': 'l',
    'humidity': 'l',
    'wind': 'w'
})

default_class = data[target].mode().iloc[0]

prediction = classify(tree, new_sample, default_class)

if prediction == 1:
    print("Classification: Play Tennis = Yes")
else:
    print("Classification: Play Tennis = No")

In [2]:
import math
from collections import Counter

# Dataset: Age, Blood Pressure, Cholesterol, Diagnosis
data = [
    [30, "High", "High", "Sick"],
    [45, "Low", "Normal", "Healthy"],
    [50, "High", "High", "Sick"],
    [35, "Low", "Normal", "Healthy"],
    [60, "High", "High", "Sick"],
    [55, "Low", "Normal", "Healthy"],
    [40, "High", "High", "Sick"],
    [25, "Low", "Normal", "Healthy"],
    [65, "High", "High", "Sick"],
    [45, "Low", "Normal", "Healthy"]
]

features = ["Age", "Blood Pressure", "Cholesterol"]
target = "Diagnosis"


def entropy(rows):
    labels = []
    for row in rows:
        labels.append(row[-1])
    counts = Counter(labels)
    total = len(labels)

    result = 0
    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result


def split_data(rows, index):
    groups = {}

    for row in rows:
        value = row[index]
        if value not in groups:
            groups[value] = []
        groups[value].append(row)

    return groups


def information_gain(rows, index):
    parent_entropy = entropy(rows)
    groups = split_data(rows, index)
    total = len(rows)

    weighted_entropy = 0

    for group in groups.values():
        weighted_entropy += (len(group) / total) * entropy(group)

    return parent_entropy - weighted_entropy


def build_tree(rows, available_features):
    labels = [row[-1] for row in rows]

    # If all examples have the same class
    if len(set(labels)) == 1:
        return labels[0]

    # If no features remain, return majority class
    if not available_features:
        return Counter(labels).most_common(1)[0][0]

    # Find feature with highest information gain
    best_index = max(available_features,key=lambda i: information_gain(rows, i))

    tree = {features[best_index]: {}}

    groups = split_data(rows, best_index)
    remaining = []
    for i in available_features:
        if i != best_index:
            remaining.append(i)

    for value, group in groups.items():
        tree[features[best_index]][value] = build_tree(group, remaining)

    return tree


# 5. Predict using the decision tree
def predict(tree, patient):
    # A diagnosis is a string, so we have reached a leaf.
    if isinstance(tree, str):
        return tree

    for feature in tree:
        feature_index = features.index(feature)
        patient_value = patient[feature_index]
        branches = tree[feature]
        if patient_value not in branches:
            return "Unknown"
        next_node = branches[patient_value]
        answer = predict(next_node, patient)
        return answer


# Main program
print("Entropy of Diagnosis:", round(entropy(data), 4))

print("\nInformation Gain:")
for i, feature in enumerate(features):
    print(feature, "=", round(information_gain(data, i), 4))

tree = build_tree(data, list(range(len(features))))

print("\nDecision Tree:")
from pprint import pprint
print("Decision Tree:")
pprint(tree, sort_dicts=False)
# Patient: Age 50, Low blood pressure, Normal cholesterol
patient = [50, "Low", "Normal"]
print("\nPrediction:", predict(tree, patient))


Entropy of Diagnosis: 1.0

Information Gain:
Age = 1.0
Blood Pressure = 1.0
Cholesterol = 1.0

Decision Tree:
Decision Tree:
{'Age': {30: 'Sick',
         45: 'Healthy',
         50: 'Sick',
         35: 'Healthy',
         60: 'Sick',
         55: 'Healthy',
         40: 'Sick',
         25: 'Healthy',
         65: 'Sick'}}

Prediction: Sick


In [3]:
import pandas as pd
import numpy as np 
import math
from pprint import pprint

df = pd.DataFrame({
    'age' : ['y','y','m','s','s','s','m','y','y','s','y','m','m','s'],
    'income' : ['high','high','high','med','low','low','low','med','low','med','med','med','high','med'],
    'student' : ['n','y','n','n','n','y','y','n','y','y','y','n','y','n'],
    'level' : ['primary','primary','primary','college','college','college','highschool','primary','primary','college','primary','highschool','highschool','college'], 
    'buys' : ['no','no','yes','yes','yes','no','yes','no','yes','yes','yes','yes','yes','yes'] 
})


df.to_csv("data.csv", index = False)

def entropy(buys): 
    c1 = np.sum(buys=='no')/len(buys)
    c2 = np.sum(buys=='yes')/len(buys)
    r = 0
    if c1 > 0 :
        r += c1 * math.log2(c1)
    if c2 > 0:
        r += c2 * math.log2(c2)
    return -r

def information_gain(data, feature): 
    parent = entropy(data['buys'])
    we = 0

    for value in data[feature].unique():
        subset = data[data[feature]==value]
        p = len(subset)/len(data)
        if p > 0 :
            we += p * entropy(subset['buys'])

    return parent - we; 

used_features = set()

def build_tree(data, features): 
    if len(data['buys'].unique()) == 1 :
        return data['buys'].iloc[0]

    if len(features)==0 : 
        return data['buys'].mode().iloc[0]

    gain = {}

    print("\nSamples reaching node:", len(data))

    for feat in features:
        gain[feat] = information_gain(data, feat)
        print(feat,"IG:", round(gain[feat], 4))
        
    best_feat = max(gain, key=gain.get)
    print("Selected attribute:", best_feat)

    used_features.add(best_feat)

    tree = {best_feat : {}}

    print("Samples per edge:", [len(subset) for _, subset in data.groupby(best_feat)])
    for value, subset in data.groupby(best_feat):
        remaining_features = [f for f in features if f != best_feat]
        subset = subset.drop(columns=[best_feat])
        tree[best_feat][value] = build_tree(subset, remaining_features)

    return tree


def classify(tree, sample, default_class):

    if not isinstance(tree, dict):
        return tree

    node = next(iter(tree))
    value = sample[node]

    if value not in tree[node]:
        return default_class

    return classify(tree[node][value], sample, default_class)


features = ['age', 'income', 'student', 'level']
default_class = df['buys'].mode().iloc[0]

tree = build_tree(df, features)
pprint(tree, sort_dicts=False)

new_sample = {
    'age': 's',
    'income': 'high',
    'student': 'y',
    'level': 'college'
}

print("Prediction:", classify(tree, new_sample, default_class))

unused = set(features) - used_features
print("Unused attributes:", unused)



Samples reaching node: 14
age IG: 0.2585
income IG: 0.067
student IG: 0.0
level IG: 0.1767
Selected attribute: age
Samples per edge: [4, 5, 5]

Samples reaching node: 5
income IG: 0.3219
student IG: 0.3219
level IG: 0.0
Selected attribute: income
Samples per edge: [2, 3]

Samples reaching node: 2
student IG: 1.0
level IG: 0.0
Selected attribute: student
Samples per edge: [1, 1]

Samples reaching node: 5
income IG: 0.571
student IG: 0.42
level IG: 0.0
Selected attribute: income
Samples per edge: [2, 1, 2]

Samples reaching node: 2
student IG: 1.0
level IG: 0.0
Selected attribute: student
Samples per edge: [1, 1]
{'age': {'m': 'yes',
         's': {'income': {'low': {'student': {'n': 'yes', 'y': 'no'}},
                          'med': 'yes'}},
         'y': {'income': {'high': 'no',
                          'low': 'yes',
                          'med': {'student': {'n': 'no', 'y': 'yes'}}}}}}
Prediction: yes
Unused attributes: {'level'}


In [5]:
import pandas as pd
import numpy as np 
import math
from pprint import pprint

df = pd.DataFrame({
    'Age': ['Young', 'Young', 'Middle', 'Senior', 'Senior', 'Senior','Middle', 'Young', 'Young', 'Senior', 'Young', 'Middle', 'Middle', 'Senior'],
    'Income': ['High', 'High', 'High', 'Medium', 'Low', 'Low','Low', 'Medium', 'Low', 'Medium', 'Medium', 'Medium','High', 'Medium'],
    'Employment': ['Stable', 'Unstable', 'Stable', 'Stable', 'Stable', 'Unstable', 'Unstable', 'Stable', 'Stable','Unstable', 'Unstable', 'Stable', 'Unstable', 'Stable'],
    'Claims': ['None', 'Many', 'None', 'Few', 'Many', 'Few','None', 'Few', 'Many', 'None', 'None', 'Few','Many', 'None'],
    'Vehicle': ['New', 'Old', 'New', 'New', 'Old', 'Old','New', 'Old', 'New', 'Old', 'New', 'Old','New', 'New'],
    'Approved': ['Yes', 'No', 'Yes', 'Yes', 'No', 'No','Yes', 'Yes', 'No', 'No', 'Yes', 'Yes','No', 'Yes']
})

df.to_csv("data.csv", index=False)

features = ['Age', 'Income', 'Employment', 'Claims', 'Vehicle']
target = 'Approved'

## c4.5 

def entropy(feature): 
    c1 = np.sum(feature=='Yes') / len(feature)
    c2 = np.sum(feature=='No') / len(feature)
    r = 0
    if c1 > 0 : 
        r += c1 * math.log2(c1)
    if c2 > 0 :
        r += c2 * math.log2(c2)

    return -r

def information_gain(data,feature): 
    parent = entropy(data[target])
    we = 0
    si = 0

    for value in data[feature].unique() :
        s = data[data[feature]==value]
        p = len(s) / len(data)
        we += p * entropy(s[target])
        if p > 0: 
            si -= p * math.log2(p)


    return (parent - we ) / si if si > 0 else 0 

used = set()
def build_tree(data,features):

    if len(data[target].unique()) == 1 :
        return data[target].iloc[0]
    if len(features) == 0 :
        return data[target].mode().iloc[0]

    gain = {}
    print("Possibilities", len(features))
    for feat in features : 
        gain[feat] = information_gain(data,feat)
        print("Node", feat, "Gain Ratio", round(gain[feat], 4))

    best_feat = max(gain, key = gain.get)
    print("Selected",best_feat)
    used.add(best_feat)
    tree = {best_feat:{}}

    print("Branches ", [len(subset) for _ , subset in data.groupby(best_feat)])
    for value , subset in data.groupby(best_feat) : 
        remaining = [x for x in features if x!=best_feat]
        subset = subset.drop(columns = [best_feat])
        tree[best_feat][value] = build_tree(subset,remaining)

    return tree

def classify(tree, sample, default_class) : 
    if not isinstance(tree, dict) :
        return tree

    node = next(iter(tree))
    value = sample[node]

    if value not in tree[node] :
        return default_class
        
    return classify(tree[node][value], sample, default_class)

new_sample = {
    'Age': 'Senior',
    'Income': 'High',
    'Employment': 'Stable',
    'Claims': 'None',
    'Vehicle': 'New'
}

default_class = df[target].mode().iloc[0]
tree = build_tree(df,features)
pprint(tree,sort_dicts=False)
print("Classification : ", classify(tree,new_sample,default_class))
print("Unused features:", set(features) - used)

Possibilities 5
Node Age Gain Ratio 0.038
Node Income Gain Ratio 0.1215
Node Employment Gain Ratio 0.13
Node Claims Gain Ratio 0.305
Node Vehicle Gain Ratio 0.13
Selected Claims
Branches  [4, 4, 6]
Possibilities 4
Node Age Gain Ratio 0.2075
Node Income Gain Ratio 1.0
Node Employment Gain Ratio 1.0
Node Vehicle Gain Ratio 0.1511
Selected Income
Branches  [1, 3]
Possibilities 4
Node Age Gain Ratio 0.1998
Node Income Gain Ratio 0.1308
Node Employment Gain Ratio 0.1909
Node Vehicle Gain Ratio 1.0
Selected Vehicle
Branches  [5, 1]
{'Claims': {'Few': {'Income': {'Low': 'No', 'Medium': 'Yes'}},
            'Many': 'No',
            'None': {'Vehicle': {'New': 'Yes', 'Old': 'No'}}}}
Classification :  Yes
Unused features: {'Employment', 'Age'}
